# Phase 3: Vector Store & RAG Implementation

**Source (PDF Section: WHAT YOU NEED TO BUILD - 3):** "Retrieval-Augmented Generation... the system must retrieve relevant past cases."

In this notebook, we will:
1. **Initialize a Vector Database** (Qdrant) in local mode.
2. **Generate Embeddings** for our cleaned tweets using a pre-trained transformer model.
3. **Index the Data** into the vector store so we can perform semantic searches.
4. **Build a Retrieval Function** that will eventually feed context to our LLM.

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from tqdm.auto import tqdm
import os

# 1. Load the clean data (we use the full processed set for the knowledge base)
DATA_PATH = "../data/train.csv" # Using the training data as our "history"
df = pd.read_csv(DATA_PATH)

# 2. Initialize the Embedding Model
# 'all-MiniLM-L6-v2' is fast, small, and perfect for the 2ms/latency goals
model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Initialize Qdrant in Local Mode (saves to a file)
client = QdrantClient(path="../data/qdrant_db")

# 4. Create a Collection
COLLECTION_NAME = "support_cases"

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

print(f"Collection '{COLLECTION_NAME}' initialized.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Collection 'support_cases' initialized.


/tmp/ipykernel_409531/2921431169.py:22: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


## Step 4: Indexing the Knowledge Base (The Upsert Loop)
**Source (PDF Section: Retrieval-Augmented Generation):** "...the system must retrieve relevant past cases."

To make our data searchable, we need to "Upsert" (Update + Insert) our tweets into the vector database. We don't just store the vectors; we store a **Payload** containing:
* **`clean_text`**: What the LLM will read to understand the past case.
* **`author_id`**: To allow brand-specific filtering.
* **`priority`**: To show the user if similar past cases were high or low priority.

**Strategy:** We will process the data in **batches**. Sending 70,000 requests one-by-one is inefficient; sending them in groups of 100 is significantly faster and more stable.

In [2]:
# Define batch size (adjust based on your RAM)
BATCH_SIZE = 100
records = df.to_dict('records')

print(f"Starting indexing of {len(records)} points...")

# Using a loop to process batches
for i in tqdm(range(0, len(records), BATCH_SIZE)):
    batch = records[i : i + BATCH_SIZE]
    
    # 1. Extract text for embedding
    texts = [str(r['clean_text']) for r in batch]
    
    # 2. Generate Embeddings (This is the heavy part)
    embeddings = model.encode(texts)
    
    # 3. Prepare PointStructs for Qdrant
    points = []
    for j, row in enumerate(batch):
        point_id = i + j  # Simple incremental ID
        points.append(PointStruct(
            id=point_id,
            vector=embeddings[j].tolist(),
            payload={
                "text": row['clean_text'],
                "brand": row['author_id'],
                "priority": int(row['priority'])
            }
        ))
    
    # 4. Upsert to Qdrant
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points
    )

print("Indexing complete! Your vector store is now a searchable knowledge base.")

Starting indexing of 71616 points...


  0%|          | 0/717 [00:00<?, ?it/s]

/tmp/ipykernel_409531/2249777226.py:32: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20100 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Indexing complete! Your vector store is now a searchable knowledge base.


## Step 5: Semantic Search Verification
Before moving to the backend, we must verify that our "Knowledge Assistant" can actually find relevant information. We will test it with a query that doesn't use the exact keywords from our labeling regex to see if it understands **semantics**.

In [4]:
# Step 5: Semantic Search Verification (Updated for Qdrant 1.10+)
test_query = "my screen is flickering and I can't see anything"
query_vector = model.encode(test_query).tolist()

# Use query_points instead of search
# The result object contains a 'points' attribute with the hits
search_response = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=3 
)

print(f"Query: {test_query}\n")
for result in search_response.points:
    print(f"Score: {result.score:.4f}")
    print(f"Brand: {result.payload['brand']}")
    print(f"Past Case: {result.payload['text']}")
    print(f"Priority: {result.payload['priority']}")
    print("-" * 20)

Query: my screen is flickering and I can't see anything

Score: 0.6516
Brand: AppleSupport
Past Case: were glad to work with you on this from the video it looks like the screen may be flickering does that accurately describe the issue
Priority: 1
--------------------
Score: 0.6191
Brand: AppleSupport
Past Case: lets get you back to using your phone as you should what ios version is running on your iphone also what is happening on your device when the flickering occurs
Priority: 0
--------------------
Score: 0.6025
Brand: AppleSupport
Past Case: whats happening with your screen provide us more details so we can help
Priority: 1
--------------------
